# Compare Results: Distillation vs From Scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marjanstoimcev/DINOv3distill/blob/main/compare_results.ipynb)

This notebook compares the results from:
1. `dinov3_distillation.ipynb` - YOLO11l-seg with DINOv3 distillation pretraining
2. `train_from_scratch.ipynb` - YOLO11l-seg trained from scratch

**Run this after completing both training notebooks.**

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

## 1. Load Metrics

In [ ]:
# Load metrics from both experiments
metrics_files = {
    "DINOv3 Distillation": "./output/finetune_seg/metrics_distillation.json",
    "From Scratch": "./output/from_scratch/metrics_scratch.json"
}

results = {}
for name, path in metrics_files.items():
    try:
        with open(path, 'r') as f:
            results[name] = json.load(f)
        print(f"✓ Loaded: {name}")
    except FileNotFoundError:
        print(f"✗ Not found: {path}")
        print(f"  Run the corresponding notebook first!")

if len(results) < 2:
    print("\n⚠️ Please run both training notebooks before comparing!")

In [ ]:
# Display raw metrics
for name, metrics in results.items():
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    for key, value in metrics.items():
        if isinstance(value, float):
            print(f"  {key}: {value:.4f}")
        else:
            print(f"  {key}: {value}")

## 2. Create Comparison Table

In [ ]:
# Create comparison dataframe
if len(results) >= 2:
    comparison_data = []
    
    for name, metrics in results.items():
        comparison_data.append({
            "Method": name,
            "Box mAP50": metrics.get("box_map50", 0),
            "Box mAP50-95": metrics.get("box_map", 0),
            "Mask mAP50": metrics.get("seg_map50", 0),
            "Mask mAP50-95": metrics.get("seg_map", 0),
        })
    
    df = pd.DataFrame(comparison_data)
    df = df.set_index("Method")
    
    print("\n" + "="*70)
    print("COMPARISON TABLE")
    print("="*70)
    print(df.to_string())
    print("="*70)

In [ ]:
# Calculate improvement
if len(results) >= 2:
    distill = results.get("DINOv3 Distillation", {})
    scratch = results.get("From Scratch", {})
    
    print("\n" + "="*70)
    print("IMPROVEMENT (Distillation vs From Scratch)")
    print("="*70)
    
    metrics_to_compare = [
        ("Box mAP50", "box_map50"),
        ("Box mAP50-95", "box_map"),
        ("Mask mAP50", "seg_map50"),
        ("Mask mAP50-95", "seg_map"),
    ]
    
    for display_name, key in metrics_to_compare:
        d_val = distill.get(key, 0)
        s_val = scratch.get(key, 0)
        
        if s_val > 0:
            improvement = ((d_val - s_val) / s_val) * 100
            abs_diff = d_val - s_val
            symbol = "↑" if improvement > 0 else "↓"
            print(f"  {display_name:15s}: {abs_diff:+.4f} ({improvement:+.2f}%) {symbol}")
        else:
            print(f"  {display_name:15s}: N/A")
    
    print("="*70)

## 3. Visualize Comparison

In [ ]:
# Bar chart comparison
if len(results) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    methods = list(results.keys())
    colors = ['#3498db', '#e74c3c']
    
    # Box mAP
    ax = axes[0]
    x = np.arange(2)
    width = 0.35
    
    map50_vals = [results[m].get("box_map50", 0) for m in methods]
    map_vals = [results[m].get("box_map", 0) for m in methods]
    
    bars1 = ax.bar(x - width/2, map50_vals, width, label='mAP50', color='#2ecc71')
    bars2 = ax.bar(x + width/2, map_vals, width, label='mAP50-95', color='#9b59b6')
    
    ax.set_ylabel('mAP Score')
    ax.set_title('Box Detection Performance', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(methods)
    ax.legend()
    ax.set_ylim(0, max(map50_vals + map_vals) * 1.2)
    
    # Add value labels
    for bar in bars1 + bars2:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9)
    
    # Mask mAP
    ax = axes[1]
    
    map50_vals = [results[m].get("seg_map50", 0) for m in methods]
    map_vals = [results[m].get("seg_map", 0) for m in methods]
    
    bars1 = ax.bar(x - width/2, map50_vals, width, label='mAP50', color='#2ecc71')
    bars2 = ax.bar(x + width/2, map_vals, width, label='mAP50-95', color='#9b59b6')
    
    ax.set_ylabel('mAP Score')
    ax.set_title('Instance Segmentation Performance', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(methods)
    ax.legend()
    ax.set_ylim(0, max(map50_vals + map_vals) * 1.2)
    
    # Add value labels
    for bar in bars1 + bars2:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9)
    
    plt.suptitle('YOLO11l-seg: DINOv3 Distillation vs Training from Scratch', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('comparison_chart.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Side-by-Side Inference Comparison

In [ ]:
# Load and display inference images side by side
from PIL import Image
import matplotlib.pyplot as plt

inference_images = {
    "DINOv3 Distillation": "./output/finetune_seg/inference_distillation.png",
    "From Scratch": "./output/from_scratch/inference_scratch.png"
}

fig, axes = plt.subplots(2, 1, figsize=(20, 10))

for ax, (name, path) in zip(axes, inference_images.items()):
    try:
        img = Image.open(path)
        ax.imshow(img)
        ax.set_title(name, fontsize=14, fontweight='bold')
        ax.axis('off')
    except FileNotFoundError:
        ax.text(0.5, 0.5, f"Image not found:\n{path}", 
                ha='center', va='center', fontsize=12)
        ax.set_title(name, fontsize=14, fontweight='bold')
        ax.axis('off')

plt.suptitle('Inference Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('inference_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Summary

In [ ]:
if len(results) >= 2:
    distill = results.get("DINOv3 Distillation", {})
    scratch = results.get("From Scratch", {})
    
    seg_improvement = ((distill.get("seg_map", 0) - scratch.get("seg_map", 0)) / scratch.get("seg_map", 1)) * 100
    
    print("\n" + "="*70)
    print("SUMMARY")
    print("="*70)
    print(f"\nDataset: TACO (1,499 images, 59 classes)")
    print(f"Model: YOLO11l-seg (27.7M params)")
    print(f"\nDINOv3 Distillation:")
    print(f"  - Pretrain epochs: {distill.get('distillation_epochs', 'N/A')}")
    print(f"  - Finetune epochs: {distill.get('finetune_epochs', 'N/A')}")
    print(f"  - Mask mAP50-95: {distill.get('seg_map', 0):.4f}")
    print(f"\nFrom Scratch:")
    print(f"  - Train epochs: {scratch.get('epochs', 'N/A')}")
    print(f"  - Mask mAP50-95: {scratch.get('seg_map', 0):.4f}")
    print(f"\n{'🎉' if seg_improvement > 0 else '📉'} Segmentation Improvement: {seg_improvement:+.2f}%")
    print("="*70)
    
    if seg_improvement > 0:
        print("\n✅ DINOv3 distillation pretraining improved performance!")
    else:
        print("\n⚠️ Consider increasing distillation epochs or adjusting hyperparameters.")